# CEG5305 Part 2, Lecture 1, session 02: the adaptation homework, in Colab

This notebook runs [`adapt_base.py`](https://costa-nus.github.io/CEG5305_GenAI/part2/lecture-1/code/adapt_base.py): one small pre-trained language model (Qwen2.5-0.5B, 494 million parameters) adapted to SST-2 sentiment classification in four ways, with the same test set and the same prompt format: `zero` (instruction only), `icl` (labelled examples in the prompt), `lora` (train a low-rank update on the same examples) and `full` (update all weights on them).

**Runtime:** *Runtime → Change runtime type → T4 GPU*. Run time per seed on a T4: TO BE MEASURED ON COLAB. `zero` has no randomness and runs once; the other methods run three seeds.

In [ ]:
# 1. A GPU must be on: Runtime -> Change runtime type -> T4 GPU
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run this cell again"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Get the base code. If the address below is not published yet, upload adapt_base.py by hand
#    (file panel on the left, Upload) before running this cell: a copy that is already here is kept.
import os
if not os.path.exists("adapt_base.py"):
    !wget -q -O adapt_base.py https://costa-nus.github.io/CEG5305_GenAI/part2/lecture-1/code/adapt_base.py || rm -f adapt_base.py
assert os.path.exists("adapt_base.py"), "adapt_base.py is not here: upload it in the file panel on the left"
!pip install -q datasets
!python adapt_base.py --help | head -30

In [ ]:
# 3. Check the setup: every line must say PASS. The first run downloads the model (about 1 GB).
#    A warning about HF_TOKEN can be ignored: no account is needed.
!python adapt_base.py --selfcheck

In [ ]:
# 4. Zero-shot: the instruction and the review, no examples. Accuracy to expect: TO BE MEASURED ON COLAB.
!python adapt_base.py --method zero --out zero.json

In [ ]:
# 5. In-context learning: 16 labelled examples in the prompt, three seeds (the seed picks the examples)
!python adapt_base.py --method icl --n 16 --out icl_n16.json

In [ ]:
# 6. LoRA: train a rank-8 update on the same 16 examples, then test with the zero-shot prompt
!python adapt_base.py --method lora --n 16 --rank 8 --out lora_n16.json

In [ ]:
# 7. Full fine-tuning: update all weights on the same 16 examples
!python adapt_base.py --method full --n 16 --out full_n16.json

## 8. Your question

Choose one question about adapting a model, for example: with the same `n` labelled examples, is it better to put them in the prompt or to train LoRA on them? Write your predictions down before you run anything. Then change **one** thing at a time: a flag (`--method`, `--n`, `--rank`, `--labels`, `--model`), or one value in `FIXED` if that value is your question (open `adapt_base.py` in the file panel on the left, double-click to edit, save). Keep the three seeds, give every run its own `--out` file, and compare differences with the spread across seeds.

In [ ]:
!python adapt_base.py --method icl --n 16 --labels flipped --out icl_n16_flipped.json   # replace with the runs for your question

In [ ]:
# 9. The results table: one line per JSON file in this folder. Below it, the training curves of lora and full.
import glob, json
import matplotlib.pyplot as plt

print(f"{'file':22s} {'method':6s} {'labels':8s} {'n':>4s} {'rank':>4s} {'mean':>6s} {'min-max':>11s}  {'per seed':18s} {'trainable':>12s} {'tokens':>7s} {'s/seed':>7s} {'GPU GB':>7s}")
curves = {}
for path in sorted(glob.glob("*.json")):
    d = json.load(open(path))
    if "results" not in d or "settings" not in d:
        continue
    r, s, runs = d["results"], d["settings"], d["results"]["runs"]
    per_seed = "/".join(f"{x:.1f}" for x in r["accuracy"])
    seconds = sum(x["seconds"] for x in runs) / len(runs)
    gpu = max((x.get("gpu_gb") or 0) for x in runs)
    print(f"{path[:-5]:22s} {s['method']:6s} {s['labels']:8s} {s['n']:4d} {str(s['rank'] or ''):>4s} {r['mean']:6.1f} {r['min']:5.1f}-{r['max']:<5.1f}  {per_seed:18s} "
          f"{runs[0]['trainable']:12,} {runs[0]['prompt_tokens']:7.0f} {seconds:7.0f} {gpu:7.2f}")
    if d["curves"]:
        curves[path[:-5]] = list(d["curves"].values())

if curves:
    fig, ax = plt.subplots(figsize=(7, 4))
    for name, per_seed in curves.items():
        steps = per_seed[0]["step"]
        mean = [sum(c["train_loss"][i] for c in per_seed) / len(per_seed) for i in range(len(steps))]
        ax.plot(steps, mean, label=f"{name} (mean of {len(per_seed)} seeds)")
    ax.set_xlabel("training step"); ax.set_ylabel("training loss (cross-entropy of the label word)")
    ax.set_yscale("log"); ax.legend(); ax.grid(alpha=.3)
    plt.show()